## Load data

In [7]:
from plot_umap import *
from plot_pca import *
import pandas as pd

hs_train = pd.read_json("../../experiment/datasets/goldenswag/golden_swag_train_synthetic_OFF_TOPIC_SAMPLES.json")
hs_train_logs = pd.read_json(
    "../../experiment/datasets/goldenswag/golden_swag_train_synthetic_OFF_TOPIC_SAMPLES_logs.json")

random_state = 114

# Extract the set of off-topic IDs for quick lookup
off_topic_ids = set(log["id"] for log in hs_train_logs.to_dict("records"))

options = [0, 1, 2, 3]

data = {}
for id, entry in hs_train.iterrows():
    context = entry["ctx"]
    correct_ending = entry["endings"][entry["label"]]
    wrong_endings = [entry["endings"][i] for i in options if i != entry["label"]]

    # Assuming the ID in hs_train is stored in entry["ind"]
    task_id = str(entry["ind"])
    is_offtopic = any(log_id.startswith(f"{task_id}-") for log_id in off_topic_ids)

    data_point = {
        "text": context + " " + correct_ending,
        "correct": 1,
        "category": entry["activity_label"],
        "task_id": id,
        "off_topic": is_offtopic
    }
    data[len(data)] = data_point
    for wrong in wrong_endings:
        data_point = {
            "text": context + " " + wrong,
            "correct": 0,
            "category": entry["activity_label"],
            "task_id": id,
            "off_topic": is_offtopic
        }
        data[len(data)] = data_point

df = pd.DataFrame.from_dict(data, orient="index")
print("Total samples:", len(df))


def filter_df(df: DataFrame, top_n_categories: int, n_samples: int) -> DataFrame:
    # Get top N categories
    top_n_categories = df['category'].value_counts().head(top_n_categories).index.tolist()
    filtered_df = df[df["category"].isin(top_n_categories)]

    # Group by task_id and sample groups rather than individual rows
    grouped = filtered_df.groupby('task_id')

    # Get list of all task_ids
    task_ids = list(grouped.groups.keys())

    # Sample task_ids (not individual rows)
    sampled_task_ids = pd.Series(task_ids).sample(n=min(round(n_samples / 4), len(task_ids)), random_state=random_state)

    # Get all rows for the sampled task_ids
    sampled_df = filtered_df[filtered_df['task_id'].isin(sampled_task_ids)]

    print(f"Number of samples={len(sampled_df)}, got {len(sampled_task_ids)} task groups")
    return sampled_df.reset_index(drop=True)


def get_random_sampled(df: DataFrame, n_samples: int) -> DataFrame:
    # Group by task_id to ensure we sample complete task groups
    grouped = df.groupby('task_id')

    # Get list of all task_ids
    task_ids = list(grouped.groups.keys())

    # Calculate how many task groups we need to sample
    # Each task group typically contains 4 rows (context + 3 endings)
    # So we divide the requested number of samples by 4
    n_task_groups = min(round(n_samples / 4), len(task_ids))

    # Randomly sample task_ids
    sampled_task_ids = pd.Series(task_ids).sample(n=n_task_groups, random_state=random_state)

    # Get all rows for the sampled task_ids
    sampled_df = df[df['task_id'].isin(sampled_task_ids)]

    print(f"Requested {n_samples} samples, got {len(sampled_df)} samples from {n_task_groups} task groups")

    return sampled_df.reset_index(drop=True)


import torch
from transformers import AutoTokenizer, AutoModel
from pandas import DataFrame
from torch import Tensor

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Load models and tokenizers
#model_finetuned = AutoModel.from_pretrained("../models/Electra_golden_swag_train/epoch35").to(device)
model_finetuned = AutoModel.from_pretrained("google/electra-base-discriminator").to(device)

tokenizer_finetuned = AutoTokenizer.from_pretrained("google/electra-base-discriminator")

model = AutoModel.from_pretrained('bert-base-uncased').to(device)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

model.train(False)
model_finetuned.train(False)


def get_embeddings(df: DataFrame) -> tuple[Tensor, Tensor]:
    # Tokenize inputs for both models
    inputs_finetuned = tokenizer_finetuned(df["text"].tolist(), padding=True, truncation=True, return_tensors="pt")
    inputs = tokenizer(df["text"].tolist(), padding=True, truncation=True, return_tensors="pt")

    # Move all input tensors to the MPS device
    inputs_finetuned = {k: v.to(device) for k, v in inputs_finetuned.items()}
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate embeddings for both models in batches
    batch_size = 64
    embeddings_finetuned = []
    embeddings_original = []

    with torch.no_grad():
        for i in range(0, len(inputs["input_ids"]), batch_size):
            # Finetuned model batch
            batch_finetuned = {k: v[i:i + batch_size] for k, v in inputs_finetuned.items()}
            batch_embeddings_finetuned = model_finetuned(**batch_finetuned, return_dict=True).last_hidden_state[:, 0, :]
            embeddings_finetuned.append(batch_embeddings_finetuned.cpu())  # Move to CPU after computation

            # Original model batch
            batch_original = {k: v[i:i + batch_size] for k, v in inputs.items()}
            batch_embeddings_original = model(**batch_original, return_dict=True).last_hidden_state[:, 0, :]
            embeddings_original.append(batch_embeddings_original.cpu())  # Move to CPU after computation

    # Concatenate all batches for both models
    embeddings_finetuned = torch.cat(embeddings_finetuned, dim=0)
    embeddings_original = torch.cat(embeddings_original, dim=0)

    # L2 Normalize each embedding vector
    embeddings_finetuned = embeddings_finetuned / torch.norm(embeddings_finetuned, dim=1, keepdim=True)
    embeddings_original = embeddings_original / torch.norm(embeddings_original, dim=1, keepdim=True)

    return embeddings_finetuned, embeddings_original


Total samples: 4880
Using device: mps


/Users/davebrunner/Documents/repositories/SelfClean/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning:

`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.

Some weights of the model checkpoint at google/electra-base-discriminator were not used when initializing ElectraModel: ['discriminator_predictions.dense.bias', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense.weight', 'discriminator_predictions.dense_prediction.weight']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a 

## 200 samples with Off-Topic 3D PCA

In [8]:
df_y = get_random_sampled(df, 200)
embeddings_finetuned, embeddings_1 = get_embeddings(df_y)

Requested 200 samples, got 200 samples from 50 task groups


In [9]:
plot_pca_3d_combined(embeddings_finetuned, embeddings_1, df_y, color_label="off_topic", model_finetuned_name="ELECTRA",
                     model_2_name="BERT")

## Top 3 categories & 500 samples UMAP 3D

In [2]:
df_1 = get_random_sampled(df, 500)
embeddings_1_finetuned, embeddings_1 = get_embeddings(df_1)

NameError: name 'get_random_sampled' is not defined

In [ ]:
neighbors = list(range(10, 56, 2))
for neighbor in neighbors:
    plot_umap_3d_compared(embeddings_1, embeddings_1_finetuned, df_1, neighbor)

## Single category & 100 samples

In [ ]:
## Single category & 1'000 samples
df_top1_100 = filter_df(df, 1, 100)
embeddings_1_100_finetuned, embeddings_1_100 = get_embeddings(df_top1_100)

In [ ]:
neighbors = list(range(5, 30, 5))
for neighbor in neighbors:
    plot_umap_compared(embeddings_1_100, embeddings_1_100_finetuned, df_top1_100, neighbor, color_label="task_id")

## Top 10 categories & 100 samples

In [ ]:
df_top10_100 = filter_df(df, 10, 100)
embeddings_10_100_finetuned, embeddings_10_100 = get_embeddings(df_top10_100)

In [ ]:
neighbors = list(range(15, 35, 5))
for neighbor in neighbors:
    plot_umap_compared(embeddings_10_100, embeddings_10_100_finetuned, df_top10_100, neighbor)

Single Samples

## Single category & 100 samples

In [ ]:
df_top1_20 = filter_df(df, 1, 100)
embeddings_1_20_finetuned, embeddings_1_20 = get_embeddings(df_top1_20)

In [ ]:
neighbors = list(range(15, 35, 5))
for neighbor in neighbors:
    plot_umap_1d_compared(embeddings_1_20, embeddings_1_20_finetuned, df_top1_20, neighbor, color_label="task_id")

## Pair of embeddings with PCA

In [ ]:
from plot_pca import *


def compare_pairs_with_pca_3d(
    df: pd.DataFrame,
    num_samples: int = 1000,
    num_splits: int = 2,
    samples_per_plot: int = 2,
    color_label: str = "task_id"
):
    """
    Automatically compare pairs of embeddings (regular vs. fine-tuned) using UMAP.

    Args:
        df: DataFrame containing the data.
        num_samples: Number of samples to randomly select from the DataFrame.
        num_splits: Number of splits to divide the embeddings into.
        samples_per_plot: Number of samples to include in each UMAP plot.
        color_label: Column name for coloring points in the UMAP plot.
    """
    # Get random samples
    df_sampled = get_random_sampled(df, num_samples)

    # Get embeddings
    embeddings_finetuned, embeddings = get_embeddings(df_sampled)

    # Split embeddings
    split_size = embeddings.size(0) // num_splits
    split_embeddings = torch.split(embeddings, split_size, dim=0)
    split_embeddings_finetuned = torch.split(embeddings_finetuned, split_size, dim=0)

    # Iterate over splits and plot
    for i in range(num_splits):
        start_idx = i * samples_per_plot
        end_idx = start_idx + samples_per_plot

        # Get the subset of the DataFrame for the current split
        df_subset = df_sampled.iloc[start_idx:end_idx]

        # Plot UMAP for the current split
        plot_pca_3d_combined(
            split_embeddings[i],
            split_embeddings_finetuned[i],
            df_subset,
            color_label=color_label,
            model_finetuned_name="ELECTRA",
            model_2_name="BERT")

In [ ]:
compare_pairs_with_pca_3d(df, num_samples=10, num_splits=4, samples_per_plot=2, color_label="task_id")